# CatBoost Mixed Features + NLP Experiment

Bu notebook, mevcut CatBoost mixed feature yapısına `mentor_feedback_text` üzerinden NLP tabanlı sayısal feature'lar eklenmiş deney sürümüdür.

Notlar:
- `TRAIN_PATH = "train.csv"`
- `TEST_PATH = "test_x.csv"`
- `academic_score` yok.
- `mentor_feedback_text` raw haliyle modele verilmez; sadece numeric NLP feature'lar kullanılır.
- Submission dosyaları `results/` klasörüne kaydedilir.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

RANDOM_STATE = 42


In [ ]:
TRAIN_PATH = "train.csv"
TEST_PATH = "test_x.csv"

TARGET = "career_success_score"
ID_COL = "student_id"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

RUN_NAME = "catboost_mixed_features_nlp"


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

display(train_df.head())
display(test_df.head())


In [ ]:
print("Train missing values:")
display(train_df.isna().sum()[train_df.isna().sum() > 0].sort_values(ascending=False))

print("\nTest missing values:")
display(test_df.isna().sum()[test_df.isna().sum() > 0].sort_values(ascending=False))


In [ ]:
def add_missing_flags(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    missing_cols = [
        "english_exam_score",
        "internship_duration_months",
        "portfolio_score",
        "github_avg_stars",
        "open_source_contribution_count",
        "linkedin_profile_score",
        "hr_interview_score"
    ]

    for col in missing_cols:
        if col in df.columns:
            df[f"{col}_was_missing"] = df[col].isna().astype(int)

    return df


def fill_missing_values(train: pd.DataFrame, test: pd.DataFrame):
    train = train.copy()
    test = test.copy()

    mean_cols = [
        "english_exam_score",
        "linkedin_profile_score",
        "hr_interview_score",
        "portfolio_score"
    ]

    median_cols = [
        "github_avg_stars",
        "open_source_contribution_count"
    ]

    for col in mean_cols:
        if col in train.columns:
            fill_value = train[col].mean()
            train[col] = train[col].fillna(fill_value)
            test[col] = test[col].fillna(fill_value)

    for col in median_cols:
        if col in train.columns:
            fill_value = train[col].median()
            train[col] = train[col].fillna(fill_value)
            test[col] = test[col].fillna(fill_value)

    # Internship duration: internship_count grubuna göre median
    col = "internship_duration_months"
    group_col = "internship_count"

    if col in train.columns and group_col in train.columns:
        global_median = train[col].median()
        group_medians = train.groupby(group_col)[col].median()

        train[col] = train.apply(
            lambda row: group_medians.get(row[group_col], global_median)
            if pd.isna(row[col]) else row[col],
            axis=1
        )

        test[col] = test.apply(
            lambda row: group_medians.get(row[group_col], global_median)
            if pd.isna(row[col]) else row[col],
            axis=1
        )

        train[col] = train[col].fillna(global_median)
        test[col] = test[col].fillna(global_median)

    # Text kolonunda NaN varsa boş string yap
    if "mentor_feedback_text" in train.columns:
        train["mentor_feedback_text"] = train["mentor_feedback_text"].fillna("")
        test["mentor_feedback_text"] = test["mentor_feedback_text"].fillna("")

    return train, test


In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Tarih/yaş ilişkili feature'lar
    df["years_since_graduation"] = df["application_year"] - df["graduation_year"]
    df["age_at_graduation"] = df["age"] - df["years_since_graduation"]
    df["is_recent_graduate"] = (df["years_since_graduation"] <= 1).astype(int)

    # Teknik skor özetleri
    technical_skill_cols = [
        "coding_score", "problem_solving_score", "data_structures_score",
        "sql_score", "machine_learning_score", "backend_score",
        "frontend_score", "cloud_score", "devops_score"
    ]

    df["technical_skill_mean"] = df[technical_skill_cols].mean(axis=1)
    df["technical_skill_std"] = df[technical_skill_cols].std(axis=1)
    df["technical_skill_min"] = df[technical_skill_cols].min(axis=1)
    df["technical_skill_max"] = df[technical_skill_cols].max(axis=1)
    df["technical_skill_range"] = (
        df["technical_skill_max"] - df["technical_skill_min"]
    )

    # Data/AI odaklı skor
    df["data_ai_score"] = df[
        [
            "sql_score",
            "machine_learning_score",
            "problem_solving_score",
            "data_structures_score",
            "coding_score"
        ]
    ].mean(axis=1)

    # Software engineering odaklı skor
    df["software_engineering_score"] = df[
        [
            "backend_score",
            "frontend_score",
            "cloud_score",
            "devops_score",
            "coding_score",
            "data_structures_score"
        ]
    ].mean(axis=1)

    # Soft skill özeti
    soft_skill_cols = [
        "communication_score",
        "teamwork_score",
        "leadership_score",
        "presentation_score",
        "linkedin_profile_score",
        "cv_quality_score",
        "hr_interview_score"
    ]

    df["soft_skill_mean"] = df[soft_skill_cols].mean(axis=1)
    df["soft_skill_std"] = df[soft_skill_cols].std(axis=1)

    # Deneyim/aktivite özeti
    experience_cols = [
        "real_client_project_count",
        "internship_count",
        "freelance_project_count",
        "hackathon_count",
        "certification_count",
        "bootcamp_count"
    ]

    df["experience_total"] = df[experience_cols].sum(axis=1)

    # Project / portfolio
    df["project_portfolio_score"] = (
        df["project_quality_score"]
        + df["portfolio_score"]
        + df["real_client_project_count"] * 5
        + df["freelance_project_count"] * 3
        + df["github_repo_count"] * 0.5
        + df["github_avg_stars"] * 0.5
        + df["open_source_contribution_count"] * 2
    )

    df["project_impact"] = (
        df["project_quality_score"] * (df["real_client_project_count"] + 1)
    )

    # Experience score
    df["experience_score"] = (
        df["internship_count"] * 10
        + df["internship_duration_months"] * 2
        + df["freelance_project_count"] * 5
        + df["real_client_project_count"] * 7
    )

    # Competition / hackathon
    df["competition_score"] = (
        df["hackathon_count"] * 3
        + df["hackathon_awards"] * 10
    )

    df["hackathon_success_rate"] = (
        df["hackathon_awards"] / (df["hackathon_count"] + 1)
    )

    # Learning activity
    df["learning_activity_score"] = (
        df["certification_count"] * 2
        + df["bootcamp_count"] * 5
    )

    # Başvuru -> mülakat funnel'ı
    df["interview_conversion_rate"] = (
        df["interviews_attended"] / (df["applications_sent"] + 1)
    )

    df["applications_without_interview"] = (
        df["applications_sent"] - df["interviews_attended"]
    )

    df["applications_per_year_after_grad"] = (
        df["applications_sent"] / (df["years_since_graduation"] + 1)
    )

    df["interview_per_year_after_grad"] = (
        df["interviews_attended"] / (df["years_since_graduation"] + 1)
    )

    df["interview_score_mean"] = df[
        ["technical_interview_score", "hr_interview_score"]
    ].mean(axis=1)

    df["weighted_interview_score"] = (
        df["technical_interview_score"] * 0.6
        + df["hr_interview_score"] * 0.4
    )

    df["interview_success_score"] = (
        df["interview_conversion_rate"] * df["weighted_interview_score"]
    )

    # Github feature'ları
    df["github_impact_score"] = (
        df["github_repo_count"]
        + df["github_avg_stars"] * 2
        + df["open_source_contribution_count"] * 3
    )

    df["github_activity"] = (
        df["github_repo_count"] * df["github_avg_stars"]
    )

    df["github_per_repo_quality"] = (
        df["github_avg_stars"] / (df["github_repo_count"] + 1)
    )

    # Görünürlük / profil gücü
    df["visibility_index"] = (
        df["linkedin_profile_score"] * 0.4
        + df["portfolio_score"] * 0.3
        + df["github_impact_score"] * 0.3
    )

    df["portfolio_visibility_score"] = (
        df["portfolio_score"] * 0.5
        + df["linkedin_profile_score"] * 0.3
        + df["cv_quality_score"] * 0.2
    )

    # Interaction feature'lar
    df["technical_x_project"] = (
        df["technical_skill_mean"] * df["project_portfolio_score"]
    )

    df["technical_x_experience"] = (
        df["technical_skill_mean"] * df["experience_score"]
    )

    df["soft_x_interview"] = (
        df["soft_skill_mean"] * df["weighted_interview_score"]
    )

    df["project_x_visibility"] = (
        df["project_portfolio_score"] * df["visibility_index"]
    )

    df["technical_x_visibility"] = (
        df["technical_skill_mean"] * df["visibility_index"]
    )

    df["internship_months_per_internship"] = (
        df["internship_duration_months"] / (df["internship_count"] + 1)
    )

    df["applications_pressure"] = (
        df["applications_sent"] / (df["interviews_attended"] + 1)
    )

    # =========================================================
    # NLP FEATURE'LARI - mentor_feedback_text üzerinden
    # =========================================================

    feedback_lower = (
        df["mentor_feedback_text"]
        .fillna("")
        .astype(str)
        .str.lower()
    )

    # Basic text length features
    df["mentor_feedback_len"] = feedback_lower.str.len()

    df["mentor_feedback_word_count"] = (
        feedback_lower
        .str.split()
        .str.len()
    )

    tech_keywords = [
        "makine öğrenimi", "veri yapıları", "veri bilimi", "veri analizi",
        "backend", "frontend", "bulut", "yazılım", "açık kaynak",
        "github", "sql", "yapay zeka", "devops", "cloud", "kodlama",
        "problem çözme", "algoritma", "mobil", "siber güvenlik", "data",
        "analist", "mühendis", "developer"
    ]

    soft_keywords = [
        "iletişim", "takım çalışması", "mülakat", "proje", "takım",
        "liderlik", "sunum", "ekip", "zaman yönetimi",
        "analitik düşünme", "yönetim", "ikna"
    ]

    positive_signals = [
        "etkileyici", "güçlü", "dikkat çekici", "dikkat çekiyor",
        "dikkat çeken", "sağlam temel", "yüksek", "başarılı",
        "harika", "potansiyel", "iyi seviyede", "mükemmel",
        "tatmin edici", "beklentileri aşıyor", "yetkin"
    ]

    negative_signals = [
        "faydalı olacaktır", "fazla pratik", "deneyim kazanması",
        "pratik yapması", "geliştirmeli", "eksik", "odaklanmalı",
        "çalışmalı", "yetersiz", "zayıf", "beklentilerin altında",
        "artırmalı", "ihtiyacı var"
    ]

    contrast_words = [
        "ancak", "fakat", "rağmen", "gösterse de",
        "bununla birlikte", "yine de", "yalnız"
    ]

    df["nlp_tech_keyword_count"] = feedback_lower.apply(
        lambda text: sum(keyword in text for keyword in tech_keywords)
    )

    df["nlp_soft_keyword_count"] = feedback_lower.apply(
        lambda text: sum(keyword in text for keyword in soft_keywords)
    )

    df["nlp_positive_signal_count"] = feedback_lower.apply(
        lambda text: sum(signal in text for signal in positive_signals)
    )

    df["nlp_negative_signal_count"] = feedback_lower.apply(
        lambda text: sum(signal in text for signal in negative_signals)
    )

    df["nlp_has_contrast"] = feedback_lower.apply(
        lambda text: int(any(word in text for word in contrast_words))
    )

    df["nlp_sentiment_balance"] = (
        df["nlp_positive_signal_count"] - df["nlp_negative_signal_count"]
    )

    df["nlp_tech_positive_interaction"] = (
        df["nlp_tech_keyword_count"] * df["nlp_positive_signal_count"]
    )

    df["nlp_tech_negative_interaction"] = (
        df["nlp_tech_keyword_count"] * df["nlp_negative_signal_count"]
    )

    df["nlp_soft_positive_interaction"] = (
        df["nlp_soft_keyword_count"] * df["nlp_positive_signal_count"]
    )

    df["nlp_soft_negative_interaction"] = (
        df["nlp_soft_keyword_count"] * df["nlp_negative_signal_count"]
    )

    df = df.replace([np.inf, -np.inf], np.nan)

    return df


In [ ]:
train_tmp = add_missing_flags(train_df)
test_tmp = add_missing_flags(test_df)

train_tmp, test_tmp = fill_missing_values(train_tmp, test_tmp)

train_fe = add_features(train_tmp)
test_fe = add_features(test_tmp)

print("Train missing after feature engineering:", train_fe.isna().sum().sum())
print("Test missing after feature engineering :", test_fe.isna().sum().sum())

print("Train shape after features:", train_fe.shape)
print("Test shape after features :", test_fe.shape)


In [ ]:
DROP_COLS = [TARGET, ID_COL, "mentor_feedback_text"]

feature_cols = [col for col in train_fe.columns if col not in DROP_COLS]

X = train_fe[feature_cols].copy()
y = train_fe[TARGET].copy()
X_test = test_fe[feature_cols].copy()

cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
cat_feature_indices = [X.columns.get_loc(col) for col in cat_cols]

print("Number of features:", len(feature_cols))
print("Categorical columns:", cat_cols)

display(X.head())


In [ ]:
cat_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 1800,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 6,
    "random_seed": RANDOM_STATE,
    "od_type": "Iter",
    "od_wait": 150,
    "verbose": 300,
    "allow_writing_files": False
}


In [ ]:
def run_catboost_cv(params, n_splits=5, seed=RANDOM_STATE):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

    oof_preds = np.zeros(len(train_fe))
    test_preds = np.zeros(len(test_fe))
    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), start=1):
        print(f"\n========== Fold {fold} ==========")

        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

        model = CatBoostRegressor(**params)

        model.fit(
            X_train,
            y_train,
            cat_features=cat_feature_indices,
            eval_set=(X_valid, y_valid),
            use_best_model=True
        )

        valid_pred = model.predict(X_valid)
        oof_preds[valid_idx] = valid_pred

        fold_rmse = mean_squared_error(y_valid, valid_pred) ** 0.5
        fold_scores.append(fold_rmse)

        print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

        test_preds += model.predict(X_test) / n_splits

    oof_rmse = mean_squared_error(y, oof_preds) ** 0.5
    oof_mse = mean_squared_error(y, oof_preds)

    print("\n========== CV RESULT ==========")
    print(f"Fold RMSE values: {[round(s, 5) for s in fold_scores]}")
    print(f"Mean RMSE: {np.mean(fold_scores):.5f}")
    print(f"Std RMSE : {np.std(fold_scores):.5f}")
    print(f"OOF RMSE : {oof_rmse:.5f}")
    print(f"OOF MSE  : {oof_mse:.5f}")

    return {
        "oof_preds": oof_preds,
        "test_preds": test_preds,
        "fold_scores": fold_scores,
        "oof_rmse": oof_rmse,
        "oof_mse": oof_mse
    }


In [ ]:
base_result = run_catboost_cv(cat_params, n_splits=5, seed=RANDOM_STATE)

test_preds = base_result["test_preds"]
oof_preds = base_result["oof_preds"]
oof_rmse = base_result["oof_rmse"]
oof_mse = base_result["oof_mse"]
fold_scores = base_result["fold_scores"]


In [ ]:
test_preds_clipped = np.clip(test_preds, 0, 100)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_preds_clipped
})

submission_path = RESULTS_DIR / f"{RUN_NAME}_submission.csv"
submission.to_csv(submission_path, index=False)

display(submission.head())
print(f"Saved: {submission_path}")


## Optional parameter experiments

Aşağıdaki hücreyi sadece ekstra CatBoost parametre denemeleri yapmak istediğinde çalıştır. En iyi sonucu `oof_mse` değerine göre seçer ve sadece best CSV dosyasını `results/best_param_submission.csv` olarak kaydeder.


In [ ]:
RUN_PARAM_EXPERIMENTS = False

if RUN_PARAM_EXPERIMENTS:
    param_grid = {
        "mix_nlp_v1_depth6_lr003": {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "iterations": 1800,
            "learning_rate": 0.03,
            "depth": 6,
            "l2_leaf_reg": 6,
            "random_seed": RANDOM_STATE,
            "od_type": "Iter",
            "od_wait": 150,
            "verbose": 300,
            "allow_writing_files": False
        },
        "mix_nlp_v2_depth5_regularized": {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "iterations": 2500,
            "learning_rate": 0.025,
            "depth": 5,
            "l2_leaf_reg": 9,
            "random_seed": RANDOM_STATE,
            "od_type": "Iter",
            "od_wait": 200,
            "verbose": 300,
            "allow_writing_files": False
        },
        "mix_nlp_v3_depth7_slow": {
            "loss_function": "RMSE",
            "eval_metric": "RMSE",
            "iterations": 3000,
            "learning_rate": 0.02,
            "depth": 7,
            "l2_leaf_reg": 7,
            "random_seed": RANDOM_STATE,
            "od_type": "Iter",
            "od_wait": 250,
            "verbose": 300,
            "allow_writing_files": False
        }
    }

    experiment_results = {}

    for name, params in param_grid.items():
        print(f"#################### {name} ####################")
        result = run_catboost_cv(params, n_splits=5, seed=RANDOM_STATE)
        experiment_results[name] = result

    summary = pd.DataFrame([
        {
            "experiment": name,
            "oof_rmse": result["oof_rmse"],
            "oof_mse": result["oof_mse"],
            "mean_fold_rmse": np.mean(result["fold_scores"]),
            "std_fold_rmse": np.std(result["fold_scores"]),
        }
        for name, result in experiment_results.items()
    ]).sort_values("oof_mse")

    display(summary)

    best_name = summary.iloc[0]["experiment"]
    best_result = experiment_results[best_name]

    best_test_preds = np.clip(best_result["test_preds"], 0, 100)

    best_submission = pd.DataFrame({
        ID_COL: test_df[ID_COL],
        TARGET: best_test_preds
    })

    best_submission_path = RESULTS_DIR / "best_param_submission.csv"
    best_submission.to_csv(best_submission_path, index=False)

    print(f"Best experiment: {best_name}")
    print(f"Best OOF RMSE: {best_result['oof_rmse']:.5f}")
    print(f"Best OOF MSE : {best_result['oof_mse']:.5f}")
    print(f"Saved: {best_submission_path}")
    display(best_submission.head())
